#### 4. Dimension → `dim_product`

(You don’t explicitly have clean product table — so derive)

#### Columns:

```text
product_id (generated OR item name)

product_name
unit
category (if derivable)
```

In [0]:
# ============================================================
# 0. CONFIG PATH
# ============================================================
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

# ============================================================
# 1. IMPORTS
# ============================================================
import json
import pyspark.sql.functions as f
from common_utils.generic_functions import  casting
from common_utils.logging import get_logger

# ============================================================
# 2. LOGGER
# ============================================================
logger = get_logger("dim_products.ipynb")
logger.info("Session for dim_products started")

# ============================================================
# 3. READ CONFIG
# ============================================================

logger.info(
    "Reading Gold configuration from %s",config_path
)
with open(config_path,"r") as file:
    config = json.load(file)

logger.info("Gold configuration loaded successfully")



# ============================================================
# 4. SOURCE / TARGET CONFIGURATION
# ============================================================
source_config = config["source"]
target_config= config["target"]

source_name = config["source"]["source_name"]
source_table = config["source"]["source_table"]
target_table = config["target"]["target_table"]
category_table = config["source"]["category_source"]

logger.info(
    "[%s] Source table: %s",
    source_name,
    source_table
)

logger.info(
    "[%s] Target table: %s",
    source_name,
    target_table
)
# ============================================================
# 5. READ SOURCE TABLE
# ============================================================

logger.info(
    "[%s] Reading order from %s",
    source_name,
    source_table
)

order_products_df = spark.read.table(source_table)
sales_df = spark.read.table(category_table)

logger.info(
    "[%s] product table loaded successfully",
    source_name
)
logger.info(
    "[%s] category column table loaded successfully",
    sales_df
)

# ============================================================
# 6. TRANSFORM CUSTOMER
# ============================================================


def transform_products(df_products,config):
    logger.info(
        "[%s] Starting dim_products transformation",
        source_name
    )
    # --------------------------------------------------------
    # 6.1 PRIMARY PRODUCT DATA
    # --------------------------------------------------------
    
    df_products = (
        order_products_df.select(f.col("id").alias("product_id"),f.col("product_name"),f.col("unit")).filter(f.col("product_id").isNotNull())
    )

    logger.info("Primary columns selected for the final")

    # --------------------------------------------------------
    # 6.2 Enrishing table for the join to get the category column
    # --------------------------------------------------------
    
    cat_df = (
        sales_df.select(f.col("product_id"),f.col("product_category").alias("category")).filter(f.col("product_id").isNotNull() & f.col("category").isNotNull()).dropDuplicates(["product_id"])
    )
    logger.info("enrichment for the category column done successfully")

    # --------------------------------------------------------
    # 6.3 joining the tables
    # --------------------------------------------------------

    df_products = df_products.join(cat_df, on="product_id",how ="left").withColumn("category",f.coalesce(f.col("category"), f.lit("unknown")))

    # --------------------------------------------------------
    # 6.3 joining the tables
    # ------------(--------------------------------------------
    df_products = (df_products.dropDuplicates(["product_id"]))
    logger.info(
        "dim_product transformation completed"
    )

    return df_products
       

# ============================================================
# 7. CREATE DIM PRODUCTS
# ============================================================

logger.info(
    "[%s] Starting dim_products creation",
    source_name
)

#creating dim products
dim_products = transform_products(order_products_df,config)


# ============================================================
# 8. VALIDATE DATA
# ============================================================
logger.info(
    "[%s] Validating dim_products",
    source_name
)

records_count=dim_products.count()
if records_count==0:
    raise ValueError("no record found in Dim_products")

duplicate_count = (
    dim_products
    .groupBy("product_id")
    .count()
    .filter(f.col("count") > 1)
    .count()
)
if duplicate_count > 0:
    raise ValueError ("Duplicae detected")


# ============================================================
# 9. DISPLAY
# ============================================================
dim_products.display()


# ============================================================
# 10. CREATE GOLD SCHEMA
# ============================================================
logger.info(
    "[%s] Creating Gold schema if required",
    source_name
)

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS 
    retail_data_platform.gold
    
          """)

# ============================================================
# 11. WRITE GOLD TABLE
# ============================================================

logger.info(
    "[%s] Writing dim_order to %s",
    source_name,
    target_table
)

(dim_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table ))

logger.info(
    "[%s] Gold table %s written successfully",
    source_name,
    target_table
)


# ============================================================
# 12. FINAL VALIDATION
# ============================================================

final_count = (
    spark.read
    .table(target_table)
    .count()
)

logger.info(
    "[%s] Final dim_products record count: %s",
    source_name,
    final_count
)

logger.info(
    "[%s] dim_products pipeline completed successfully",
    source_name
)







